#Lab: Fine-tuning a Transformer-based Machine Translation Model (English–Vietnamese)
In this lab, you will learn how to fine-tune a pre-trained Transformer-based neural machine translation (NMT) model for English–Vietnamese translation.
Transformer architectures such as mBART, MarianMT, and mT5 have achieved strong multilingual performance.
By fine-tuning them on a bilingual corpus, we can adapt the models for improved domain-specific English–Vietnamese translation quality.

##Dataset for English–Vietnamese Machine Translation

In this lab, we use a bilingual dataset from the Hugging Face Hub — specifically, the ncduy/mt-en-vi
 dataset — which contains millions of aligned English–Vietnamese sentence pairs.
Each sample consists of a source sentence in English ("en") and its corresponding translation in Vietnamese ("vi").

Machine translation datasets like this are typically parallel corpora, where each sentence in one language is precisely aligned with its equivalent in another.
Such data is essential for training sequence-to-sequence Transformer models, enabling them to learn semantic mapping between two languages, handle grammar and word order differences, and generate fluent target-language outputs.

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from typing import Dict, List, Optional
import datasets

In [2]:
from datasets import load_dataset

dataset = load_dataset("ncduy/mt-en-vi")

print("\nDataset structure:")
print(dataset)

print("\nDataset info (before sampling):")
print(f"Train samples: {len(dataset['train']):,}")
print(f"Validation samples: {len(dataset['validation']):,}")
print(f"Test samples: {len(dataset['test']):,}")
# You can modify the size of the train set
sampled_train = dataset["train"].train_test_split(test_size=0.1, seed=42)["test"]

dataset["train"] = sampled_train

print("\nAfter sampling 10% of original train:")
print(f"New train samples: {len(dataset['train']):,}")
print(f"Validation samples: {len(dataset['validation']):,}")
print(f"Test samples: {len(dataset['test']):,}")

print("\nExample samples:")
for i in range(3):
    print(dataset["train"][i])

README.md:   0%|          | 0.00/3.43k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B /  597MB            

train.csv: downloading bytes:           |  0.00B            

valid.csv: reconstructing file:   0%|          |  0.00B / 2.45MB            

valid.csv: downloading bytes:           |  0.00B            

test.csv: reconstructing file:   0%|          |  0.00B / 2.43MB            

test.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2884451 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11316 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11225 [00:00<?, ? examples/s]


Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['en', 'vi', 'source'],
        num_rows: 2884451
    })
    validation: Dataset({
        features: ['en', 'vi', 'source'],
        num_rows: 11316
    })
    test: Dataset({
        features: ['en', 'vi', 'source'],
        num_rows: 11225
    })
})

Dataset info (before sampling):
Train samples: 2,884,451
Validation samples: 11,316
Test samples: 11,225

After sampling 10% of original train:
New train samples: 288,446
Validation samples: 11,316
Test samples: 11,225

Example samples:
{'en': '"education and leading me to providing service to the Republic."', 'vi': '"nơi giáo dục và dìu dắt tôi để tôi có thể phụng sự nền Cộng Hòa"', 'source': 'OpenSubtitles v2018'}
{'en': 'Because of the big, bad Fizzle Bomber?', 'vi': 'Có phải vì Kẻ Ném Bom Xì Xèo xấu xa?', 'source': 'OpenSubtitles v2018'}
{'en': 'The author of numerous collections of poems, short stories, young adult novels, and seven novels, Mars is one of the m

##Building the Dataset and DataLoader for Machine Translation

In a machine translation (MT) task, the model learns to convert a source sentence (e.g., English) into a target sentence (e.g., Vietnamese).
To efficiently train such models, it is crucial to prepare the data in a structured format that the Transformer architecture can process.

We first define a custom Dataset class that reads the English–Vietnamese sentence pairs from the dataset and applies tokenization using a pretrained tokenizer (such as mBART or MarianMT).
This class transforms each example into a pair of tensors:

- input_ids for the source (English) text

- labels for the target (Vietnamese) text
A DataLoader is used to efficiently batch, shuffle, and pad these examples during training.
The DataLoader combines multiple tokenized samples into mini-batches of equal tensor size, enabling parallelized computation on GPU and stabilizing gradient updates.

The overall data pipeline ensures that:

1. Sentences are properly tokenized and truncated to a fixed length.

2. Padded sequences are aligned for efficient batching.

3. The model receives consistent, language-paired input for each training step.

This structured data preparation process is essential for fine-tuning any sequence-to-sequence Transformer model on bilingual translation tasks.

In [8]:

class TranslationDataset(Dataset):
    """Dataset for eng-vi translation task"""
    """
    Args:
        data: A HuggingFace Dataset object
        tokenizer: Tokenizer from the transformers library
        max_length: The maximum sequence length
        src_lang: Source language (default: 'en')
        tgt_lang: Target language (default: 'vi')
    """
    def __init__(
        self,
        data: datasets.Dataset,
        tokenizer: AutoTokenizer,
        max_length: int = 256,
        src_lang: str = "en",
        tgt_lang: str = "vi"
    ):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        item = self.data[idx]
        src_text = item[self.src_lang]
        tgt_text = item[self.tgt_lang]

        # Tokenize source text
        src_encoded = self.tokenizer(
            # TODO
            src_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # Tokenize target text
        tgt_encoded = self.tokenizer(
            # TODO
            text_target=tgt_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            'input_ids': src_encoded['input_ids'].squeeze(0),
            'attention_mask': src_encoded['attention_mask'].squeeze(0),
            'labels': tgt_encoded['input_ids'].squeeze(0),
            'decoder_attention_mask': tgt_encoded['attention_mask'].squeeze(0)
        }

In [9]:
import torch
from torch.utils.data import DataLoader
from typing import Dict, List, Tuple

def collate_fn(batch: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    """
    Custom collate function to combine batch data.

    Args:
        batch: List of dictionaries from Dataset.__getitem__

    Returns:
        Dictionary containing stacked tensors
    """

    return {
        'input_ids': torch.stack([item['input_ids'] for item in batch]),
        'attention_mask': torch.stack([item['attention_mask'] for item in batch]),
        'labels': torch.stack([item['labels'] for item in batch]),
        'decoder_attention_mask': torch.stack([item['decoder_attention_mask'] for item in batch])
    }


def create_dataloaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size: int = 32,
    num_workers: int = 4,
    pin_memory: bool = True
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """
    Create DataLoaders for the train, validation, and test datasets.

    Args:
        train_dataset: Training dataset
        val_dataset: Validation dataset
        test_dataset: Test dataset
        batch_size: Batch size (default: 32)
        num_workers: Number of workers for the DataLoader (default: 4)
        pin_memory: Whether to pin memory for GPU usage (default: True)

    Returns:
        Tuple[DataLoader, DataLoader, DataLoader]: train_loader, val_loader, test_loader
    """

    train_loader = DataLoader(
        # TODO
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=collate_fn
    )

    val_loader = DataLoader(
        # TODO
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=collate_fn
    )

    test_loader = DataLoader(
        # TODO
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=collate_fn
    )

    return train_loader, val_loader, test_loader

In [10]:
tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-vi")
print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"Vocab size: {len(tokenizer)}")


# ===== CREATE DATASETS =====
print("\n" + "=" * 60)
print("CREATING PYTORCH DATASETS")
print("=" * 60)

max_length = 128
batch_size = 8

train_dataset = TranslationDataset(
    dataset['train'],
    tokenizer,
    max_length
)

val_dataset = TranslationDataset(
    dataset['validation'],
    tokenizer,
    max_length
)

test_dataset = TranslationDataset(
    dataset['test'],
    tokenizer,
    max_length
)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

train_loader, val_loader, test_loader = create_dataloaders(
  train_dataset,
  val_dataset,
  test_dataset,
  batch_size=batch_size
)

print("\n" + "=" * 60)
print("DATALOADERS CREATED")
print("=" * 60)

print(f"Train loader length: {len(train_loader)}")
print(f"Validation loader length: {len(val_loader)}")
print(f"Test loader length: {len(test_loader)}")



Tokenizer loaded: MarianTokenizer
Vocab size: 53685

CREATING PYTORCH DATASETS
Train dataset size: 288446
Validation dataset size: 11316
Test dataset size: 11225

DATALOADERS CREATED
Train loader length: 36056
Validation loader length: 1415
Test loader length: 1404


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## Fine-tuning a Transformer for English–Vietnamese Translation

This code provides a complete training pipeline for fine-tuning a Transformer-based sequence-to-sequence model (such as mBART, MarianMT, or mT5) on the English–Vietnamese translation task.
It defines modular functions to train, evaluate, and generate translations using PyTorch and Hugging Face Transformers.

The workflow includes:

- train_epoch() – performs one epoch of model training with gradient clipping and learning rate scheduling.

- evaluate() – computes average loss on the validation or test set without gradient updates.

- generate_translations() – produces translation samples from the test data for qualitative evaluation.

- finetune() – integrates all components into a complete fine-tuning loop with automatic model saving, progress logging, and final evaluation.

To use this script:

1. Replace the # TODO sections with AutoModelForSeq2SeqLM.from_pretrained(model_name) and AutoTokenizer.from_pretrained(model_name).

2. Provide preprocessed train_loader, val_loader, and test_loader.

3. Run the script to start fine-tuning your translation model.

In [11]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, get_linear_schedule_with_warmup
from tqdm import tqdm
import time


def train_epoch(model, train_loader, optimizer, scheduler, device, epoch, log_interval=100):
    """
    Train the model for one epoch.

    Args:
        model: The mBART model
        train_loader: Training DataLoader
        optimizer: Optimizer
        scheduler: Learning rate scheduler
        device: Device (cuda/cpu)
        epoch: Current epoch number
        log_interval: Log every specified number of steps

    Returns:
        avg_loss: Average loss for the epoch
    """
    model.train()
    total_loss = 0
    start_time = time.time()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")

    for step, batch in enumerate(pbar, 1):
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        # Update progress bar
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'avg_loss': f'{total_loss/step:.4f}',
            'lr': f'{scheduler.get_last_lr()[0]:.2e}'
        })

        if step % log_interval == 0:
            elapsed = time.time() - start_time
            print(f"\n[Epoch {epoch} - Step {step}/{len(train_loader)}]")
            print(f"  Loss: {loss.item():.4f}")
            print(f"  Avg Loss: {total_loss/step:.4f}")
            print(f"  LR: {scheduler.get_last_lr()[0]:.2e}")
            print(f"  Time: {elapsed:.2f}s")

    avg_loss = total_loss / len(train_loader)
    return avg_loss


def evaluate(model, eval_loader, device, split_name="Validation"):
    """
    Evaluate the model on the validation or test set.

    Args:
        model: The mBART model
        eval_loader: Evaluation DataLoader
        device: Device (cuda/cpu)
        split_name: Name of the dataset split (Validation/Test)

    Returns:
        avg_loss: Average loss
    """
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in tqdm(eval_loader, desc=f"Evaluating {split_name}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_loss += outputs.loss.item()

    avg_loss = total_loss / len(eval_loader)
    return avg_loss


def generate_translations(model, test_loader, tokenizer, device, num_samples=5):
    """
    Generate translations for a number of samples from the test set.

    Args:
        model: Opus model
        test_loader: Test DataLoader
        tokenizer: Tokenizer
        device: Device
        num_samples: Number of samples to generate

    Returns:
        List of tuples (source, reference, prediction)
    """

    model.eval()
    results = []

    with torch.no_grad():
        for batch in test_loader:
            if len(results) >= num_samples:
                break

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Generate
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=128,
                num_beams=5,
                early_stopping=True
            )

            # Decode
            for i in range(min(input_ids.size(0), num_samples - len(results))):
                source = tokenizer.decode(input_ids[i], skip_special_tokens=True)
                reference = tokenizer.decode(labels[i], skip_special_tokens=True)
                prediction = tokenizer.decode(generated_ids[i], skip_special_tokens=True)

                results.append((source, reference, prediction))

    return results


def finetune(
    train_loader,
    val_loader,
    test_loader,
    model_name="facebook/mbart-large-50",
    num_epochs=2,
    learning_rate=5e-5,
    warmup_steps=500,
    log_interval=100,
    save_path=".content/model"
):
    """
    Fine-tune the model for the English–Vietnamese translation task.

    Args:
        train_loader: Training DataLoader
        val_loader: Validation DataLoader
        test_loader: Test DataLoader
        model_name: Model name (mBART, MarianMT, mT5, etc.)
        num_epochs: Number of epochs (default: 2)
        learning_rate: Learning rate (default: 5e-5)
        warmup_steps: Number of warmup steps (default: 500)
        log_interval: Log every N steps (default: 100)
        save_path: Path to save the model (default: ./mbart_finetuned)
    """

    # Setup device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Load model and tokenizer
    print(f"\nLoading model: {model_name}")
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name) # TODO
    tokenizer = AutoTokenizer.from_pretrained(model_name) # TODO
    model.to(device)

    # Setup optimizer và scheduler
    optimizer = AdamW(model.parameters(), lr=learning_rate)
    total_steps = len(train_loader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    print(f"\nTraining Configuration:")
    print(f"  Epochs: {num_epochs}")
    print(f"  Learning rate: {learning_rate}")
    print(f"  Total steps: {total_steps}")
    print(f"  Warmup steps: {warmup_steps}")
    print(f"  Train batches: {len(train_loader)}")
    print(f"  Val batches: {len(val_loader)}")
    print(f"  Test batches: {len(test_loader)}")

    # Training loop
    print("\n" + "="*60)
    print("STARTING TRAINING")
    print("="*60)

    best_val_loss = float('inf')

    for epoch in range(1, num_epochs + 1):
        print(f"\n{'='*60}")
        print(f"EPOCH {epoch}/{num_epochs}")
        print(f"{'='*60}")

        # Train
        train_loss = train_epoch(
            model, train_loader, optimizer, scheduler, device, epoch, log_interval
        )

        print(f"\n[Epoch {epoch} Training Complete]")
        print(f"  Average Train Loss: {train_loss:.4f}")

        # Evaluate
        print(f"\n{'='*60}")
        print(f"VALIDATION - EPOCH {epoch}")
        print(f"{'='*60}")

        val_loss = evaluate(model, val_loader, device, "Validation")

        print(f"\n[Epoch {epoch} Validation Complete]")
        print(f"  Validation Loss: {val_loss:.4f}")

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            print(f"\n✓ New best model! Saving to {save_path}")
            model.save_pretrained(save_path)
            tokenizer.save_pretrained(save_path)

        print(f"\n{'='*60}")
        print(f"EPOCH {epoch} SUMMARY")
        print(f"{'='*60}")
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss: {val_loss:.4f}")
        print(f"  Best Val Loss: {best_val_loss:.4f}")

    # Test evaluation
    print("\n" + "="*60)
    print("FINAL TEST EVALUATION")
    print("="*60)

    # Load best model
    print(f"\nLoading best model from {save_path}")
    model = AutoModelForSeq2SeqLM.from_pretrained(save_path)
    model.to(device)

    test_loss = evaluate(model, test_loader, device, "Test")

    print(f"\n[Test Evaluation Complete]")
    print(f"  Test Loss: {test_loss:.4f}")

    # Generate sample translations
    print("\n" + "="*60)
    print("SAMPLE TRANSLATIONS")
    print("="*60)

    samples = generate_translations(model, test_loader, tokenizer, device, num_samples=5)

    for i, (source, reference, prediction) in enumerate(samples, 1):
        print(f"\n[Sample {i}]")
        print(f"Source:     {source}")
        print(f"Reference:  {reference}")
        print(f"Prediction: {prediction}")

    print("\n" + "="*60)
    print("TRAINING COMPLETED!")
    print("="*60)
    print(f"✓ Best model saved to: {save_path}")
    print(f"✓ Final Test Loss: {test_loss:.4f}")

    return model, tokenizer


if __name__ == "__main__":
  model, tokenizer = finetune(
        train_loader,
        val_loader,
        test_loader
    )

# TODO FINETUNE YOUR TRANSLATION MODEL

Using device: cuda

Loading model: facebook/mbart-large-50


config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.44GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/519 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.44GB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]


Training Configuration:
  Epochs: 2
  Learning rate: 5e-05
  Total steps: 72112
  Warmup steps: 500
  Train batches: 36056
  Val batches: 1415
  Test batches: 1404

STARTING TRAINING

EPOCH 1/2




Epoch 1:   0%|          | 0/36056 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 1:   0%|          | 0/36056 [00:12<?, ?it/s, loss=20.5937, avg_loss=20.5937, lr=1.00e-07]

Epoch 1:   0%|          | 1/36056 [00:12<123:49:38, 12.36s/it, loss=20.5937, avg_loss=20.5937, lr=1.00e-07]

Epoch 1:   0%|          | 1/36056 [00:13<123:49:38, 12.36s/it, loss=16.7788, avg_loss=18.6863, lr=2.00e-07]

Epoch 1:   0%|          | 2/36056 [00:13<58:15:49,  5.82s/it, loss=16.7788, avg_loss=18.6863, lr=2.00e-07] 

Epoch 1:   0%|          | 2/36056 [00:14<58

KeyboardInterrupt: 

In [ ]:
# Do resource có hạn nên tới đây ngưng ạ

In [12]:
text = "Hello, how are you today?"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

inputs = tokenizer(text, return_tensors='pt', max_length=128, truncation=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

# Generate translation
print("Translating...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=128,
        num_beams=5,
        early_stopping=True
    )

# Decode
translation = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Hiển thị kết quả
print("="*50)
print(f"English:    {text}")
print(f"Vietnamese: {translation}")
print("="*50)

Translating...


NameError: name 'model' is not defined